In [1]:
import torch
import pandas as pd
def cov(thetas, sigmas, N, ATT):

    ub = thetas + 1.96 * sigmas/ torch.sqrt(torch.tensor(N))
    lb = thetas - 1.96 * sigmas / torch.sqrt(torch.tensor(N))
    coverage = torch.mean( ( (ATT >= lb) & (ATT <= ub) ).float() , 0 )
    interval_length = torch.mean(ub - lb,0)
    return {"Coverage":coverage,
            "interval_length": interval_length}

In [5]:
N = 2000
folder = "results_new"
# List of propensity models
models = ['logistic', 'truncated_logistic', 'truncated_adv']

# Create lists to store results
model_names = []
rmse_linear_list = []
sd_linear_list = []
rmse_riesz_list = []
sd_riesz_list = []
mea_linear_list = []
mea_riesz_list = []
coverage_linear_list = []
coverage_riesz_list = []
int_length_linear_list = []
int_length_riesz_list = []

# Read and compute RMSE, SD, Coverage, and Interval Length for each model
for model in models:
    file_suffix = f"N:{N}_{model}"

    pred_theta = torch.load(f"{folder}/{file_suffix}_pred_theta.pt")
    pred_sig = torch.load(f"{folder}/{file_suffix}_pred_sig.pt")
    ATT = torch.load(f"{folder}/{file_suffix}_ATT_calculations.pt")["ATT"]

    rows = []

    # Define method indices and names: (index in pred_theta, method name)
    method_info = [
        (1, "Linear"),
        (2, "LASSO Riesz"),
        (3, "RF Riesz"),
        (4, "Net Riesz")
    ]

    for idx, method_name in method_info:
        est = pred_theta[:, idx]
        sig = pred_sig[:, idx]

        bias = (est - ATT).mean().item()
        rmse = torch.sqrt(torch.mean((est - ATT) ** 2)).item()
        cov_result = cov(est, sig, N, ATT)
        coverage = cov_result["Coverage"].item()
        int_len = cov_result["interval_length"].item()

        rows.append([method_name, bias, rmse, coverage, int_len])

    # Display results in a DataFrame
    df = pd.DataFrame(rows, columns=["Method", "Bias", "RMSE", "Coverage", "Interval Length"])
    
    print(model)
    print(df.to_string(index=False))


logistic
     Method      Bias     RMSE  Coverage  Interval Length
     Linear  0.011748 0.152668      0.70         0.323021
LASSO Riesz  0.078394 0.167950      0.96         0.665968
   RF Riesz  0.075024 0.787339      0.96         1.459942
  Net Riesz -0.021074 0.834935      0.94         2.593856
truncated_logistic
     Method      Bias     RMSE  Coverage  Interval Length
     Linear -0.062966 0.137714      0.79         0.342889
LASSO Riesz  0.002275 0.116698      0.98         0.596370
   RF Riesz -0.008089 0.115154      0.98         0.599245
  Net Riesz -0.015156 0.108497      0.98         0.612073
truncated_adv
     Method      Bias     RMSE  Coverage  Interval Length
     Linear -0.060863 0.142656      0.81         0.341873
LASSO Riesz  0.004437 0.122133      0.96         0.596914
   RF Riesz -0.010491 0.121667      0.96         0.600962
  Net Riesz -0.013189 0.114937      0.99         0.612889


In [6]:
N = 2000
folder = "results"
# List of propensity models
models = ['logistic', 'truncated_logistic', 'truncated_adv']

# Create lists to store results
model_names = []
rmse_linear_list = []
sd_linear_list = []
rmse_riesz_list = []
sd_riesz_list = []
mea_linear_list = []
mea_riesz_list = []
coverage_linear_list = []
coverage_riesz_list = []
int_length_linear_list = []
int_length_riesz_list = []

# Read and compute RMSE, SD, Coverage, and Interval Length for each model
for model in models:
    file_suffix = f"N:{N}_{model}"

    pred_theta = torch.load(f"{folder}/{file_suffix}_pred_theta.pt")
    pred_sig = torch.load(f"{folder}/{file_suffix}_pred_sig.pt")
    ATT = torch.load(f"{folder}/{file_suffix}_ATT_calculations.pt")["ATT"]

    rows = []

    # Define method indices and names: (index in pred_theta, method name)
    method_info = [
        (1, "Linear"),
        (2, "LASSO Riesz"),
        (3, "RF Riesz"),
        (4, "Net Riesz")
    ]

    for idx, method_name in method_info:
        est = pred_theta[:, idx]
        sig = pred_sig[:, idx]

        bias = (est - ATT).mean().item()
        rmse = torch.sqrt(torch.mean((est - ATT) ** 2)).item()
        cov_result = cov(est, sig, N, ATT)
        coverage = cov_result["Coverage"].item()
        int_len = cov_result["interval_length"].item()

        rows.append([method_name, bias, rmse, coverage, int_len])

    # Display results in a DataFrame
    df = pd.DataFrame(rows, columns=["Method", "Bias", "RMSE", "Coverage", "Interval Length"])
    
    print(model)
    print(df.to_string(index=False))


logistic
     Method     Bias     RMSE  Coverage  Interval Length
     Linear 0.011748 0.152668      0.70         0.323021
LASSO Riesz 0.066528 0.148359      0.97         0.604818
   RF Riesz 0.095798 0.781541      0.96         1.342419
  Net Riesz 0.023954 0.660346      0.93         2.216125
truncated_logistic
     Method      Bias     RMSE  Coverage  Interval Length
     Linear -0.062966 0.137714      0.79         0.342889
LASSO Riesz  0.002517 0.114855      0.99         0.592055
   RF Riesz  0.000365 0.115300      0.98         0.596384
  Net Riesz -0.012389 0.110565      0.98         0.598783
truncated_adv
     Method      Bias     RMSE  Coverage  Interval Length
     Linear -0.060863 0.142656      0.81         0.341873
LASSO Riesz  0.004812 0.120236      0.97         0.591864
   RF Riesz -0.000860 0.121662      0.95         0.596325
  Net Riesz -0.011434 0.117199      0.99         0.598923
